In [6]:
import pandas as pd
import os
from utils import get_df_metrics, get_heatmap, load_json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report


In [7]:
def return_binary_classification(result):
    if "CLASSIFICATION: NOT KEY CASE" in result:
        return 0
    else:
        return 1



def transalte_importance_score(importance):
    if importance == "1":
        return 1
    else:
        return 0

def get_df(output_dir = './results_facts_openai/', keep_classes = [1,2,3,4]):
    results = os.listdir(output_dir)

    data = []
    for result in results:
        if result == ".DS_Store":
            continue
        result_data = load_json(os.path.join(output_dir, result))
        if int(result_data["importance"]) not in keep_classes:
            continue
        data.append(
            {
                'importance_level':transalte_importance_score(result_data["importance"]),
                "classification": return_binary_classification(
                    result_data["result"]["output"]
                ),
                "ground_truth": transalte_importance_score(result_data["importance"]),
            }
        )

    df = pd.DataFrame(data)
    return df


def get_metrics_binary(df):
    # Key Case vs. All
    # Ground truth: 1 for key case, 0 for not key case
    y_true = df['ground_truth']
    # Model predictions: 1 for key case, 0 for not key case
    y_pred = df['classification']

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="binary")
    recall = recall_score(y_true, y_pred, average="binary")
    f1 = f1_score(y_true, y_pred, average="binary")

    # Classification report for additional insights
    report = classification_report(y_true, y_pred, target_names=["Not Key Case", "Key Case"], output_dict=True)
    rreport = classification_report(y_true, y_pred, target_names=["Not Key Case", "Key Case"])

    # Extracting micro and macro F1 scores
    micro_f1 = report["accuracy"]  # Micro F1 is equivalent to accuracy in binary classification
    macro_f1 = report["macro avg"]["f1-score"]

    # Presenting results
    metrics = {

        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Micro F1": micro_f1,
        "Macro F1": macro_f1,
    }
    return metrics, rreport


In [8]:
output_dir = '/Users/ahmed/Desktop/msc-24/TND/key_case_binary_classification/pre_cutoff_analysis/react_1/results_openai_gpt4-mini/'
df = get_df(output_dir)
metrics, report = get_metrics_binary(df)
metrics

{'Accuracy': 0.56,
 'Precision': 0.3442622950819672,
 'Recall': 0.84,
 'F1 Score': 0.4883720930232558,
 'Micro F1': 0.56,
 'Macro F1': 0.5512035903712771}

In [9]:
print(report)

              precision    recall  f1-score   support

Not Key Case       0.90      0.47      0.61       375
    Key Case       0.34      0.84      0.49       125

    accuracy                           0.56       500
   macro avg       0.62      0.65      0.55       500
weighted avg       0.76      0.56      0.58       500



In [7]:
df = get_df(output_dir, keep_classes=[1,2])
metrics, report = get_metrics_binary(df)
metrics

{'Accuracy': 0.556,
 'Precision': 0.5357142857142857,
 'Recall': 0.84,
 'F1 Score': 0.6542056074766355,
 'Micro F1': 0.556,
 'Macro F1': 0.51704693781653}

In [8]:
df = get_df(output_dir, keep_classes=[1,3])
metrics, report = get_metrics_binary(df)
metrics

{'Accuracy': 0.596,
 'Precision': 0.5645161290322581,
 'Recall': 0.84,
 'F1 Score': 0.6752411575562701,
 'Micro F1': 0.596,
 'Macro F1': 0.5704248115823678}

In [9]:
df = get_df(output_dir, keep_classes=[1,4])
metrics, report = get_metrics_binary(df)
metrics

{'Accuracy': 0.812,
 'Precision': 0.7954545454545454,
 'Recall': 0.84,
 'F1 Score': 0.8171206225680934,
 'Micro F1': 0.812,
 'Macro F1': 0.8118524923540056}